In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tqdm
import sys
import os

In [3]:
sys.path.append('.')

In [4]:
from utils.datasets import SimpleSet
from utils.dbs import FusekiDB
from pathlib import Path
from rdflib.plugins.stores.sparqlstore import SPARQLStore
import rdflib

In [12]:
base_set = SimpleSet(base_dir=Path("../example_dataset_and_queries/knowledge_graph"))
db = FusekiDB(
    id="test",
    base_dir=Path("./scratch/fuseki_test"),
    build_dir=Path("../jena-datatensor"),
    dataset=base_set,
)
with db:
    test_query = """
    PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
    PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
    PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
    PREFIX ocs_papers: <https://w3id.org/ocs/ont/papers/>
    PREFIX dcterms: <http://purl.org/dc/terms/>
    PREFIX fabio: <http://purl.org/spar/fabio/>
    PREFIX schema: <http://schema.org/>
    PREFIX fn: <http://www.w3.org/2005/xpath-functions#>
    PREFIX frbr: <http://purl.org/vocab/frbr/core#>

    SELECT ?conferenceName (fn:round(?min, 3) AS ?score)  ?counted
    WHERE {
        {
            SELECT ?conference ?conferenceName (dta:std(?o) AS ?stdEmbedding) (COUNT(?o) AS ?counted)
            WHERE {
                ?paper a fabio:ResearchPaper ;
                    ocs_papers:hasWordEmbedding ?o ;
                    frbr:realization ?realization .

                ?realization a fabio:ConferencePaper ;
                    frbr:partOf ?conference .

                ?conference a fabio:ConferenceProceedings ;
                    schema:name ?conferenceName .
            }
            GROUP BY ?conference ?conferenceName
        }
        BIND(dtf:min(-1, ?stdEmbedding) AS ?min)
    }
    ORDER BY DESC(?min)
    LIMIT 5
    """
    results = db.query(test_query)
    print(results)

2026-03-04 10:53:49,071 - INFO - Building docker image jena-datatensor from ../jena-datatensor
2026-03-04 10:53:49,347 - INFO - Loading dataset into Fuseki server from ../example_dataset_and_queries/knowledge_graph/_complete.ttl
2026-03-04 10:53:49,355 - WARNING - DB directory scratch/fuseki_test/db already exists, removing locks if any
2026-03-04 10:53:49,360 - INFO - Stopping server with container name fuseki_benchmarks_test
2026-03-04 10:53:49,384 - ERROR - Command failed with return code 1
2026-03-04 10:53:49,404 - INFO - Starting Fuseki server with container name fuseki_benchmarks_test on port 3031
2026-03-04 10:53:51,416 - INFO - Database setup complete, entering context manager
2026-03-04 10:53:51,425 - INFO - Running SPARQL query: 
    PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
    PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
    PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
    PREFIX ocs_papers: <https://w3id.org/ocs/ont/papers/>
    PREFIX dcterms: <

                                      conferenceName  score  counted
0  Second International Conference On Research In...  0.188       47
1  11Th International Symposium Advances In Artif...  0.188       30
2  10Th International Symposium Advances In Artif...  0.186       28
3  International Conference On Research In Manage...  0.185       29
4  14Th International Workshop On Computational O...  0.179       26


In [13]:
results

,conferenceName,score,counted
0,Second International Conference On Research In...,0.188,47
1,11Th International Symposium Advances In Artif...,0.188,30
2,10Th International Symposium Advances In Artif...,0.186,28
3,International Conference On Research In Manage...,0.185,29
4,14Th International Workshop On Computational O...,0.179,26
